# 1. 실전 예상문제 (회귀)
제공된 학습용 데이터(house_price_train.csv)는 주택의 다양한 특성 정보와 실제 판매 가격(SlaePrice)을 기록한 자료이다. 해당 데이터를 기반으로 주택 가격을 예측하는 회귀 모델을 개발하고, 가장 우수한 모델을 평가 데이터(elec_test.csv)에 적용하여 주택 가격을 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함

[제공 데이터]
1. house_price_train.csv : 학습용 데이터 (1,500건)
2. house_price_test.csv : 평가용 데이터 (1,195건, SalePrice 컬럼 미제공)

[데이터 컬럼]
1. Id : 주택 식별자 (고유 번호)
2. OverallQual : 전반적 마감 품질 등급
3. GarageCars : 차고 수용 차량 수
4. GarageArea : 차고 면적 (제곱 피트)
5. YearBuilt : 건축 연도
6. YearRemoveAdd : 리모델링 연도
7. KitchenQual : 주방 품질 등급
8. .... 기타 여러가지 특성
9. SalePrice : 주택 판매 가격

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred
3. 예측 결과 개수 : 1,195




In [66]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../data/house_price_train.csv')
test = pd.read_csv('../data/house_price_test.csv')


# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.

# print(train.info())
# print(test.info())

# print(train.isnull().sum())
# print(test.isnull().sum())

# print(train['LotFrontage'].head(10))
train['LotFrontage'] = train['LotFrontage'].fillna(train['LotFrontage'].mean())
# print(train.isnull().sum())

X = train.drop(['Id','SalePrice'], axis =1)
y = train['SalePrice']
test = test.drop(['Id'],axis=1)
# print(X.shape, y.shape)

X_full = pd.concat([X, test], axis=0)
# print(X_full)

X_full = pd.get_dummies(X_full)
# print(X_full)

X_train = X_full[:train.shape[0]]
test = X_full[train.shape[0]:]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, random_state=0)
# print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# 0.84 후반~0.85 중반(채택택)
from sklearn.ensemble import RandomForestRegressor
# 튜닝 안 하는 게 나은 듯
model = RandomForestRegressor()
model.fit(X_train, y_train)


# 0.78
# from xgboost import XGBRegressor
# model = XGBRegressor()
# model.fit(X_train, y_train)

# 0.8482
# from lightgbm import LGBMRegressor
# model = LGBMRegressor()
# model.fit(X_train, y_train)


# rmse 랑은 r2 스코어 쓰자
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
print(result)
result.to_csv('result_advanced.csv', index=False)


28461.63837184409 0.8552134032541143
               pred
0     303315.137542
1     140768.981902
2     250852.244952
3     171922.652887
4     119688.965441
...             ...
1190   82305.278071
1191   82305.278071
1192  177406.447811
1193  115301.271736
1194  234504.581614

[1195 rows x 1 columns]


#1.풀이 (여러 모델의 활용)

In [ ]:
import pandas as pd

train = pd.read_csv('../data/house_price_train.csv')
test = pd.read_csv('../data/house_price_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Neighborhood'].value_counts())
#print(test['Neighborhood'].value_counts())
#print(train['KitchenQual'].value_counts())
#print(test['KitchenQual'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['SalePrice'], axis=1)
y = train['SalePrice']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['Id'], axis=1)
#print(X_full.shape)

# 결측치 처리
X_full['LotFrontage'] = X_full['LotFrontage'].fillna(X_full['LotFrontage'].mean())

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
# 모델 여러 개 테스트 해볼 때는 random_state를 넣어주기
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, random_state=0)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# (1) Randomforest 활용 모델 학습
#from sklearn.ensemble import RandomForestRegressor
#model = RandomForestRegressor()
#model.fit(X_train, y_train)

# (2) XGBoost 활용 모델 학습
#from xgboost import XGBRegressor
#model = XGBRegressor()
#model.fit(X_train, y_train)

# (3) lightGBM 활용 모델 학습
from lightgbm import LGBMRegressor
model = LGBMRegressor()
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
Note: you may need to restart the kernel to use updated packages.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000138 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 554
[LightGBM] [Info] Number of data points in the train set: 1200, number of used features: 28
[LightGBM] [Info] Start training from score 178802.054688
28992.924407280232 0.849757564927183


c:\Users\YSB\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 지정된 파일을 찾을 수 없습니다
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\YSB\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\YSB\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\YSB\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1024, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\YSB\AppData\Local

#1.풀이 (하이퍼파라미터 튜닝)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Neighborhood'].value_counts())
#print(test['Neighborhood'].value_counts())
#print(train['KitchenQual'].value_counts())
#print(test['KitchenQual'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['SalePrice'], axis=1)
y = train['SalePrice']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['Id'], axis=1)
#print(X_full.shape)

# 결측치 처리
X_full['LotFrontage'] = X_full['LotFrontage'].fillna(X_full['LotFrontage'].mean())

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, random_state=0)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# lightGBM 활용 모델 학습 - 하이퍼파라미터 튜닝
from lightgbm import LGBMRegressor
# verbose=-1: 경고나 기타 정보 빼고 결과만 보기
model = LGBMRegressor(n_estimators=100, max_depth=6,random_state=0, verbose=-1)
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

28448.08189412833 0.8553512961469888


# 2. 실전 예상문제 (분류)
제공된 학습용 데이터(phone_price_train.csv)는 휴대폰의 다양한 속성을 포함하고 있다. 이를 바탕으로 휴대폰의 등급 가격을 예측하는 다중 분류 모델을 개발하고, 가장 우수한 모델을 평가 데이터(phone_price_test.csv)에 적용하여 가격 등급 예측 결과를 도출하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 F1 Score (macro) 평가지표에 따라 평가함

[제공 데이터]
1. car_price_train.csv : 학습용 데이터 (2,000건)
2. car_price_test.csv : 평가용 데이터 (1,000건, price_grade 컬럼 미제공)

[데이터 컬럼]
1. Blue : 블루투스 지원 여부 (0: 미지원, 1: 지원)
2. clock_speed : 프로세서 속도 (GHz)
3. fc : 전면 카메라 화소 (MP)
4. int_memory : 내장 메모리 (GB)
5. m_dep : 휴대폰 두께 (cm)
6. talk_time : 연속 통화 가능 시간 (시간)
7. touch_screen : 터치 스크린 여부
8. ... 기타 여러가지 특성
9. price_range : 가격 범주 (0: 저가, 1: 중저가, 2: 중고가, 3: 고가)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred (0~3)
3. 예측 결과 개수 : 1,000




In [109]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../data/phone_price_train.csv')
test = pd.read_csv('../data/phone_price_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.
# print(train.info())
# print(test.info())

# print(train.isnull().sum())
# print(test.isnull().sum())

# print(train['price_range'].value_counts())

X = train.drop('price_range', axis=1)
y = train['price_range']

X_full = pd.concat([X, test], axis=0)
#print(X_full)

# 여기선 안 해도 되나??
X_full = pd.get_dummies(X_full)
#print(X_full)

X_train = X_full[:train.shape[0]]
test = X_full[train.shape[0]:]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, random_state=0)
# print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# 0.84~0.87, 평균 0.86대
# from sklearn.ensemble import RandomForestClassifier
# model = RandomForestClassifier()
# model.fit(X_train, y_train)

# 0.91
# from xgboost import XGBClassifier
# model = XGBClassifier()
# model.fit(X_train, y_train)

# 0.92(채택)
from lightgbm import LGBMClassifier
# 튜닝 후 0.94
model = LGBMClassifier(n_estimators=100, max_depth=8, random_state=0, verbose=-1)
model.fit(X_train, y_train)

from sklearn.metrics import f1_score, accuracy_score
y_pred = model.predict(X_val)
f1 = f1_score(y_val, y_pred, average='macro')
acc = accuracy_score(y_val, y_pred)
print(f1, acc)

y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
print(result)
result.to_csv('result_advanced_2.csv', index=False)



0.9400382427602074 0.94
     pred
0       3
1       3
2       2
3       3
4       1
..    ...
995     2
996     1
997     0
998     2
999     2

[1000 rows x 1 columns]


#1.풀이

In [5]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/phone_price_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/phone_price_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# X, Y 데이터 셋 분리
X = train.drop(['price_range'], axis=1)
y = train['price_range']

X_full = pd.concat([X, test], axis=0)
#print(X_full.shape)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y, random_state=10)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)


# (1) Randomforest 활용 모델 학습
#from sklearn.ensemble import RandomForestClassifier
#model = RandomForestClassifier()
#model.fit(X_train, y_train)

# (2) XGBoost 활용 모델 학습
from xgboost import XGBClassifier
model = XGBClassifier()
model.fit(X_train, y_train)

# (3) lightGBM 활용 모델 학습
#from lightgbm import LGBMClassifier
#model = LGBMClassifier()
#model.fit(X_train, y_train)

# f1_score, Accuracy 활용 평가
from sklearn.metrics import f1_score, accuracy_score
y_pred = model.predict(X_val)
f1 = f1_score(y_val, y_pred, average='macro')
acc = accuracy_score(y_val, y_pred)
print(f1, acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

0.9105605407489783 0.91
